# Vietnamese News Text Classification - DL Draft

Notebook này dùng để phân tích nhanh cấu trúc dữ liệu và ghi lại ý tưởng Deep Learning trước khi refactor thành pipeline trong `src/`.

## 1. Dataset

Dataset có dạng mỗi class là một folder riêng trong `data/Train_Full` và `data/Test_Full`. Mục tiêu là dự đoán topic của mỗi file tin tức tiếng Việt.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
TRAIN_DIR = DATA_DIR / 'Train_Full'
TEST_DIR = DATA_DIR / 'Test_Full'

print((DATA_DIR / 'Stats.txt').read_text(encoding='utf-8'))

In [ ]:
def count_files(root: Path) -> pd.DataFrame:
    rows = []
    for class_dir in sorted(path for path in root.iterdir() if path.is_dir()):
        rows.append({'topic': class_dir.name, 'files': len(list(class_dir.glob('*.txt')))})
    return pd.DataFrame(rows)

train_counts = count_files(TRAIN_DIR).assign(split='train')
test_counts = count_files(TEST_DIR).assign(split='test')
counts = pd.concat([train_counts, test_counts], ignore_index=True)
counts

In [ ]:
counts.pivot(index='topic', columns='split', values='files').plot(kind='bar', figsize=(12, 4), title='Files per topic')

## 2. DL Baseline Idea

Baseline refactor trong `src/` dùng PyTorch:

```text
raw text -> clean/tokenize -> vocab ids -> EmbeddingBag(mean) -> Linear -> ReLU -> Dropout -> Linear -> topic
```

`EmbeddingBag` là baseline nhanh cho text classification vì không cần pad sequence dài. Sau khi baseline ổn có thể nâng cấp sang CNN, LSTM hoặc Transformer.

In [ ]:
# Quick smoke test from notebook. Run from week3/DL.
# !python src/main.py --epochs 1 --limit-per-class 200

## 3. Training Process and Results

Sau khi chạy `python src/main.py`, pipeline lưu quá trình train vào `output/history.csv` và kết quả tốt nhất vào `output/metrics.json`.

In [ ]:
history_path = PROJECT_ROOT / 'output' / 'history.csv'
if history_path.exists():
    history = pd.read_csv(history_path)
    display(history)
    history[['train_loss', 'test_loss', 'test_accuracy', 'test_macro_f1']].plot(figsize=(10, 4), title='Training history')
else:
    print('Run python src/main.py first to generate output/history.csv')

In [ ]:
import json

metrics_path = PROJECT_ROOT / 'output' / 'metrics.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    print('Best epoch:', metrics['best_epoch'])
    print('Best metrics:', metrics['best_metrics'])
    print('Vocab size:', metrics['vocab_size'])
else:
    print('Run python src/main.py first to generate output/metrics.json')